In [1]:
import pandas as pd
df_raw = pd.read_csv("marketing_campaign_data_messy.csv")
df1 = df_raw.copy()

In [27]:
#STEP 1 : cleaning the headers
df1.columns = df1.columns.str.strip().str.lower().str.replace(" ", "_")

In [29]:
#STEP 2 : Cleaning of the column "spend" by removing the "$"
df1["spend"] = df1["spend"].str.replace("$", "", regex = False)
df1["spend"] = df1["spend"].astype(float)
df1["spend"].head(10)

In [ ]:
#STEP 3 : standardizing the names of the channels
df1["channel"] = df1["channel"].str.lower().str.strip().str.replace(r"[_-]", "", regex = True)
df1["channel"] = df1["channel"].replace({"gogle" : "google ads", "facebok" : "facebook"})

In [5]:
#STEP 4 : Standardizing the date format
df1["start_date_clean"] = pd.to_datetime(df1["start_date"], format = "mixed", dayfirst = True, errors = "coerce")
df1["end_date"] = pd.to_datetime(df1["end_date"], format = "mixed", dayfirst = True, errors = "coerce")
df1["start_date_clean"].isna().sum()

np.int64(0)

In [30]:
#STEP 5 : cleaning the campaign_tag
df1["campaign_tag"] = df1["channel"].map({"tiktok" : "TI",
                                         "facebook" : "FA",
                                         "email" : "EM",
                                         "instagram" : "IN",
                                         "google ads" : "GA"})

In [7]:
df1["channel"].unique()

array(['tiktok', 'facebook', 'email', 'instagram', 'google ads', nan],
      dtype=object)

In [8]:
#STEP 6 : replacing start_date by start_date_clean and droping start_clean_date
df1["start_date"] = df1["start_date_clean"]
df1 = df1.drop(columns = ["start_date_clean"])

In [9]:
#STEP 7 : Removing empty column clicks
df1.columns.duplicated() #Show which column is a duplicate ex : [clicks, date, clicks, campaign_id] becomes [False, False, True, False]
df1 = df1.loc[:, ~df1.columns.duplicated(keep = "first")] 
#df1.loc[:, df1.columns.duplicated()] this would select only the columns that are duplicates.  
#By adding "~" we reverse it so we have only the columns with no duplicates. We keep only original columns

In [31]:
#STEP 8 : standardizing the expected output for the "active" column
df1["active"] = df1["active"].astype(str).str.lower().str.strip()
df1["active"] = df1["active"].map({
    "y" : True,
    "yes" : True,
    "1" : True,
    "true" : True,
    "no" : False,
    "n" : False,
    "0" : False,
    "false" : False})

In [32]:
#STEP 9 : Deciding what to do with null values
df1["channel"] = df1["channel"].fillna("unknown")
df1["campaign_tag"] = df1["campaign_tag"].fillna("unknown")

In [33]:
df1.head(50)

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,tiktok,16795,197,102.82,20.0,True,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,facebook,1860,30,24.33,1.0,False,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,email,77820,843,1323.39,51.0,False,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,tiktok,55886,2019,2180.38,135.0,True,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,facebook,7265,169,252.44,30.0,True,FA
5,CMP-00006,Q4_BlackFriday_CMP-00006,2023-10-15,2023-10-28,instagram,83386,2643,2697.03,NaN,True,IN
6,CMP-00007,Q3_Launch_CMP-00007,2023-10-07,2023-10-23,facebook,38194,1135,1232.76,178.0,True,FA
7,CMP-00008,Q4_Launch_CMP-00008,2023-05-23,2023-05-28,instagram,88498,1173,865.70,127.0,True,IN
8,CMP-00009,Q4_BlackFriday_CMP-00009,2023-03-23,2023-04-01,google ads,45131,1179,1046.18,104.0,True,GA
9,CMP-00010,Q2_Winter_CMP-00010,2023-03-21,2023-04-01,email,61263,1153,1623.56,NaN,False,EM


In [34]:
df1.to_csv("marketing_campaign_data_clean.csv", index=False, encoding="utf-8-sig")